# EDA: новая логика определения дефолтности

## Логика дефолта

Компания признаётся **дефолтной**, если у неё возникает **любой один** ФП или СФП
из списка для её сегмента (не обязательно комбинация — достаточно одного фактора):

| Сегмент | ФП (факторы проблемности) | СФП (существенные) |
|---------|--------------------------|--------------------|
| **ККБ** | 13, 13.1, 15.1.3.1 | 15.2.1, 15.2.2, 15.2.1.1, 15.2.1.2 |
| **МСБ** | 13, 15.1.3 | 15.2, 15.2.1.1 |
| **МкБ** | 13, 15.1.2 | 15.2, 15.2У, 15.2.1У |

### Ключевые принципы анализа

1. **Дата дефолта** = дата первого появления определяющего фактора у данной компании.
2. **Для анализа** берутся только ФП, возникшие **строго до** даты дефолта —
   чтобы понять, какие факторы **предшествовали** дефолту.
3. **Определяющие факторы исключены** из набора предикторов —
   они задают целевую переменную и не могут одновременно быть предикторами.
4. **Для недефолтных** компаний учитываются все ФП за весь период.

### Источники данных

- `crm_last_version.csv` — выгрузка CRM с факторами проблемности
- `ref_book.csv` — справочник ФП/СФП (наименования, зоны ответственности)

## 0. Импорт и конфигурация

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from itertools import combinations

# Подавляем предупреждения, чтобы вывод был чистым
warnings.filterwarnings("ignore")

# Расширенное отображение таблиц в Jupyter
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)  # полные наименования без обрезки

# Настройка шрифтов для графиков (кириллица)
plt.rcParams.update({"font.family": "sans-serif",
                      "font.sans-serif": ["Arial", "DejaVu Sans"],
                      "axes.unicode_minus": False})

# ── Пути к файлам на сервере ──
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

# ── Период анализа ──
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")
CUTOFF    = DATE_TO  # референсная дата для недефолтных компаний

# ── Минимальное окно наблюдения ──
# Дефолты, произошедшие раньше чем через MIN_OBS_MONTHS от начала периода,
# исключаются: для них нет достаточной истории ФП-предикторов.
MIN_OBS_MONTHS = 3

# ── Маппинг X_AREA_RESP → укрупнённый сегмент бизнеса ──
# В CRM компании разбиты по зонам ответственности (X_AREA_RESP).
# Мы объединяем их в 3 аналитических сегмента.
SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

# Маппинг X_AREA_RESP → ЗО для ФП/СФП (как в справочнике ref_book.csv)
ZO_MAP = {
    "ДМСБ (микро)":   "ДМ (микро)",
    "ДМСБ (малый)":   "ДМСБ (малый)",
    "ДМСБ (средний)": "ДМСБ (средний)",
    "ДКБ":            "ДКБ",
}

# Порядок вывода сегментов
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]

# ══════════════════════════════════════════════════════════════════
# ЛОГИКА ДЕФОЛТА — предложена бизнесом
# ══════════════════════════════════════════════════════════════════
# Для каждого сегмента перечислены номера факторов, ЛЮБОЙ из которых
# означает наступление дефолта. FP и SFP разделены, чтобы при
# проверке учитывать не только номер, но и тип фактора.
# ══════════════════════════════════════════════════════════════════
DEFAULT_RULES = {
    "ККБ": {
        "FP":  {"13", "13.1", "15.1.3.1"},
        "SFP": {"15.2.1", "15.2.2", "15.2.1.1", "15.2.1.2"},
    },
    "МСБ": {
        "FP":  {"13", "15.1.3"},
        "SFP": {"15.2", "15.2.1.1"},
    },
    "МкБ": {
        "FP":  {"13", "15.1.2"},
        "SFP": {"15.2", "15.2У", "15.2.1У"},
    },
}

# Объединённые множества (FP + SFP) для быстрых проверок по номеру
DEFAULT_ALL = {seg: rules["FP"] | rules["SFP"] for seg, rules in DEFAULT_RULES.items()}

# Фильтр по источникам CRM (применяется, если колонка VAL существует)
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    """Приводит ИНН к единому строковому формату (10 или 12 знаков).
    Убирает '.0' от Excel, ведущие нули, дополняет до стандартной длины."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("Параметры заданы.")
print(f"Период: {DATE_FROM:%d.%m.%Y} — {DATE_TO:%d.%m.%Y}")
print(f"\nПравила дефолта:")
for seg in SEG_ORDER:
    rules = DEFAULT_RULES[seg]
    print(f"  {seg}: FP={sorted(rules['FP'])}  |  SFP={sorted(rules['SFP'])}")

## 1. Загрузка и подготовка данных

Загружаем CRM-выгрузку, применяем фильтры и подготавливаем данные для анализа:
- Фильтр по источникам (если доступен)
- Нормализация ИНН
- Фильтр по периоду
- Маппинг сегментов
- **Улучшенная дедупликация** с учётом `TYPE_FP` — ФП и СФП с одним номером
  считаются разными событиями

In [ ]:
# ── Загрузка CRM ──
# dtype=str — читаем всё как строки, чтобы не потерять ведущие нули ИНН
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()  # убираем пробелы из названий колонок
print(f"Загружено: {len(df):,} строк × {df.shape[1]} колонок")

# ── Фильтр по источникам ──
# В build_dataset используется фильтр по колонке VAL (источник записи).
# Если колонка есть — применяем, если нет — пропускаем с предупреждением.
if "VAL" in df.columns:
    before_src = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам ({', '.join(ALLOWED_SOURCES)}): "
          f"{len(df):,} строк (было {before_src:,})")
else:
    print(f"⚠ Колонка 'VAL' не найдена — фильтр по источникам не применён")

# ── Нормализация ИНН ──
df["inn"] = df["X_INN"].apply(normalize_inn)

# ── Преобразование даты ──
# dayfirst=True — формат ДД.ММ.ГГГГ (российский стандарт)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

# ── Фильтр по периоду ──
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(df):,} строк")

# ── Нормализация TYPE_FP ──
# В выгрузке значения могут быть латиницей (FP/SFP) — приводим к кириллице
df["TYPE_FP"] = df["TYPE_FP"].replace({"FP": "ФП", "SFP": "СФП"})

# ── Нормализация номера фактора ──
df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

# ── Маппинг сегментов ──
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

# Проверка: есть ли X_AREA_RESP, не попавшие в маппинг
unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print(f"\nНемаппированные X_AREA_RESP (исключены):")
    print(unmapped.to_string())
df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

# ── Удаление строк без номера фактора / ИНН ──
before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

# ── Улучшенная дедупликация ──
# Добавляем TYPE_FP к ключу дедупликации: ФП и СФП с одним номером в одну
# дату — это два разных события (разная тяжесть).
before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + FP + TYPE + DATE): "
      f"{len(df):,} (удалено {before_dedup - len(df):,})")

print(f"\nУникальных ИНН: {df['inn'].nunique():,}")
print(f"Уникальных номеров факторов: {df['fp_num'].nunique():,}")
print(f"\nРаспределение по сегментам:")
for seg in SEG_ORDER:
    n = df[df['segment'] == seg]['inn'].nunique()
    print(f"  {seg}: {n:,} ИНН")

## 2. Проверка наличия факторов из правил в данных

Прежде чем применять логику дефолта, убеждаемся, что **все** перечисленные бизнесом
номера факторов реально встречаются в выгрузке CRM для соответствующих сегментов.

При проверке учитываем **и номер, и тип** (FP-номера ищем среди TYPE_FP=ФП,
SFP-номера — среди TYPE_FP=СФП).

Если фактор не найден — ищем **похожие** номера (по совпадению префикса),
чтобы выявить расхождения в формате записи.

In [ ]:
all_fp_in_data = set(df["fp_num"].dropna().unique())

print(f"{'Сегмент':<8} {'Тип':<6} {'Фактор':<12} {'Найден?':<10} {'Записей в сегменте'}")
print("=" * 65)

missing = []
for seg in SEG_ORDER:
    rules = DEFAULT_RULES[seg]
    seg_df = df[df["segment"] == seg]
    # Проверяем FP-номера среди записей с TYPE_FP=ФП,
    # SFP-номера — среди записей с TYPE_FP=СФП
    for role, type_val in [("FP", "ФП"), ("SFP", "СФП")]:
        for fp in sorted(rules[role]):
            cnt = len(seg_df[(seg_df["fp_num"] == fp) & (seg_df["TYPE_FP"] == type_val)])
            exists = "Да" if cnt > 0 else "НЕТ"
            print(f"{seg:<8} {role:<6} {fp:<12} {exists:<10} {cnt:,}")
            if cnt == 0:
                missing.append((seg, role, fp))

if missing:
    print(f"\n⚠ {len(missing)} фактор(ов) НЕ НАЙДЕНО в данных:")
    for seg, role, fp in missing:
        # Нечёткий поиск: ищем номера, совпадающие по префиксу
        similar = sorted([f for f in all_fp_in_data
                          if f.startswith(fp) or fp.startswith(f)])
        print(f"   {seg}/{role}/{fp}  →  похожие: {similar if similar else '—'}")
    print("\n   Возможные причины: другой формат записи, фактор не возникал,")
    print("   фактор относится к другому сегменту.")
else:
    print("\n✅ Все факторы из правил дефолта найдены в данных.")

## 3. Определение дефолта

Для каждого ИНН в каждом сегменте:
1. Находим **все записи** с определяющими факторами (с учётом TYPE_FP)
2. `default_date` = **самая ранняя** дата из найденных
3. `default_flag = 1`, если хотя бы один определяющий фактор найден

Также анализируем, **какой именно фактор** чаще всего становится первым —
это показывает, через какой «канал» компании чаще всего входят в дефолт.

In [ ]:
# Находим записи с определяющими факторами:
# FP-номера проверяем только среди TYPE_FP=ФП, SFP — среди СФП
def_records = []
for seg in SEG_ORDER:
    rules = DEFAULT_RULES[seg]
    seg_df = df[df["segment"] == seg]
    mask_fp  = (seg_df["fp_num"].isin(rules["FP"]))  & (seg_df["TYPE_FP"] == "ФП")
    mask_sfp = (seg_df["fp_num"].isin(rules["SFP"])) & (seg_df["TYPE_FP"] == "СФП")
    def_records.append(seg_df[mask_fp | mask_sfp])

df_def_events = pd.concat(def_records, ignore_index=True)

# Агрегация до уровня компании:
# default_date = самая ранняя дата определяющего фактора
defaults = (
    df_def_events
    .groupby(["inn", "segment"])
    .agg(
        default_date=("dt", "min"),
        first_factor=("fp_num", "first"),
        n_def_factors=("fp_num", "nunique"),
    )
    .reset_index()
)
defaults["default_flag"] = 1

# Реестр ВСЕХ компаний: LEFT JOIN с дефолтами
companies = df[["inn", "segment"]].drop_duplicates()
companies = companies.merge(
    defaults[["inn", "segment", "default_flag", "default_date"]],
    on=["inn", "segment"], how="left"
)
companies["default_flag"] = companies["default_flag"].fillna(0).astype(int)

# ── Сводка по сегментам ──
print(f"{'Сегмент':<8} {'Всего ИНН':<12} {'Дефолтных':<12} {'Вер. дефолта'}")
print("=" * 45)
for seg in SEG_ORDER:
    seg_c = companies[companies["segment"] == seg]
    n_total = len(seg_c)
    n_def = seg_c["default_flag"].sum()
    pct = n_def / n_total * 100 if n_total > 0 else 0
    print(f"{seg:<8} {n_total:<12,} {n_def:<12,} {pct:.2f}%")

total = len(companies)
total_def = companies["default_flag"].sum()
print(f"{'ИТОГО':<8} {total:<12,} {total_def:<12,} {total_def/total*100:.2f}%")

In [ ]:
# ── Какой фактор чаще всего становится первым (определяющим)? ──
# Это важно для бизнеса: через какой «канал» компании входят в дефолт.
first_factors = (
    df_def_events
    .sort_values("dt")
    .groupby(["inn", "segment"])
    .first()
    .reset_index()
)

print("Первый определяющий фактор (по частоте появления):\n")
for seg in SEG_ORDER:
    seg_first = first_factors[first_factors["segment"] == seg]
    print(f"{seg}:")
    vc = seg_first["fp_num"].value_counts()
    for fp, cnt in vc.items():
        tp = seg_first[seg_first["fp_num"] == fp]["TYPE_FP"].mode().iloc[0]
        print(f"  {fp:<12} ({tp})  {cnt:>6,}  ({cnt/len(seg_first)*100:.1f}%)")
    print()

## 3.1 Минимальное окно наблюдения

**Проблема:** если компания уходит в дефолт 01.01.2023 (начало нашего периода),
у нас **нет** истории ФП до этой даты — мы не видим, какие факторы привели к дефолту.

**Решение:** ввести минимальный интервал от начала периода до даты дефолта.

Ниже оцениваем потерю при разных окнах, затем **применяем** выбранное
(`MIN_OBS_MONTHS = 3`): компании с ранним дефолтом **полностью исключаются**
из анализа (ни как дефолтные, ни как недефолтные — чтобы не загрязнять
контрольную группу).

In [ ]:
# Варианты минимального окна (месяцев от DATE_FROM)
windows = [
    ("Без ограничения", 0),
    ("≥ 3 месяца", 3),
    ("≥ 6 месяцев", 6),
    ("≥ 12 месяцев", 12),
]

# Все дефолтные компании с датой дефолта
def_with_date = companies[companies["default_flag"] == 1].copy()

print("Потеря дефолтных компаний при введении минимального окна наблюдения:\n")
header = f"{'Окно':<22}"
for seg in SEG_ORDER:
    header += f"  {seg:>12}"
header += f"  {'ИТОГО':>12}"
print(header)
print("=" * 75)

for label, months in windows:
    # Минимальная допустимая дата дефолта
    min_date = DATE_FROM + pd.DateOffset(months=months)
    row = f"{label:<22}"
    total_kept = 0
    for seg in SEG_ORDER:
        seg_def = def_with_date[def_with_date["segment"] == seg]
        all_n = len(seg_def)
        kept = len(seg_def[seg_def["default_date"] >= min_date])
        lost = all_n - kept
        pct_lost = lost / all_n * 100 if all_n > 0 else 0
        row += f"  {kept:>5,} / {all_n:>5,} (-{pct_lost:.0f}%)"
        total_kept += kept
    all_total = len(def_with_date)
    total_lost = all_total - total_kept
    # ИТОГО не считается через kept от total, считаем напрямую
    total_kept_real = len(def_with_date[def_with_date["default_date"] >= min_date])
    total_lost_real = all_total - total_kept_real
    pct_t = total_lost_real / all_total * 100 if all_total > 0 else 0
    row += f"  {total_kept_real:>5,} / {all_total:>5,} (-{pct_t:.0f}%)"
    print(row)

print(f"\nДата начала периода: {DATE_FROM:%d.%m.%Y}")
print(f"Минимальные допустимые даты дефолта:")
for label, months in windows:
    min_d = DATE_FROM + pd.DateOffset(months=months)
    print(f"  {label}: дефолт не ранее {min_d:%d.%m.%Y}")

# ══════════════════════════════════════════════════════════════════
# ПРИМЕНЯЕМ ВЫБРАННОЕ ОКНО (MIN_OBS_MONTHS)
# ══════════════════════════════════════════════════════════════════
MIN_OBS_DATE = DATE_FROM + pd.DateOffset(months=MIN_OBS_MONTHS)

# ИНН компаний, дефолт которых произошёл слишком рано
early_default_inns = companies[
    (companies["default_flag"] == 1) & (companies["default_date"] < MIN_OBS_DATE)
]["inn"].tolist()

n_early = len(early_default_inns)
print(f"\n{'='*70}")
print(f"ПРИМЕНЯЕМ ФИЛЬТР: минимальное окно = {MIN_OBS_MONTHS} мес.")
print(f"Дефолт допускается не ранее {MIN_OBS_DATE:%d.%m.%Y}")
print(f"Исключено ранних дефолтов: {n_early} ИНН")

# Полностью удаляем эти ИНН из всех таблиц
companies = companies[~companies["inn"].isin(early_default_inns)].copy()
defaults = defaults[~defaults["inn"].isin(early_default_inns)].copy()
df = df[~df["inn"].isin(early_default_inns)].copy()

n_total = len(companies)
n_def = int(companies["default_flag"].sum())
print(f"\nПосле фильтра:")
print(f"  Компаний: {n_total:,}  |  Дефолтных: {n_def:,}  |  Вер. дефолта: {n_def/n_total*100:.2f}%")
for seg in SEG_ORDER:
    sc = companies[companies['segment'] == seg]
    nd = sc['default_flag'].sum()
    print(f"  {seg}: {len(sc):,} ИНН, {nd:,} деф. ({nd/len(sc)*100:.2f}%)")

## 4. Временной фильтр ФП (anti-data-leakage)

**Ключевой шаг для корректности анализа.**

Для дефолтных компаний мы оставляем **только те ФП, которые возникли строго ДО
даты дефолта**. Это предотвращает data leakage: ФП, появившиеся после дефолта,
не могли его «предсказать» и не должны учитываться.

Для недефолтных компаний — все ФП до конца периода (`CUTOFF`).

Дополнительно: **определяющие факторы** (из `DEFAULT_RULES`) **исключаются**
из набора предикторов. Они задают целевую переменную (`default_flag`)
и не могут одновременно быть предикторами — иначе Lift и корреляции будут
тавтологически завышены.

In [ ]:
# Присоединяем default_date к каждой записи ФП (через реестр компаний)
fp_all = df[["inn", "segment", "fp_num", "TYPE_FP", "dt"]].copy()
fp_all = fp_all.merge(
    companies[["inn", "segment", "default_flag", "default_date"]],
    on=["inn", "segment"], how="inner"
)

# Референсная дата: для дефолтных = default_date, для недефолтных = CUTOFF
fp_all["ref_date"] = fp_all["default_date"].fillna(CUTOFF)

# ГЛАВНЫЙ ФИЛЬТР: оставляем ФП, возникшие СТРОГО ДО референсной даты
before_filter = len(fp_all)
fp_filtered = fp_all[fp_all["dt"] < fp_all["ref_date"]].copy()
print(f"ФП до временного фильтра:    {before_filter:,}")
print(f"ФП после временного фильтра: {len(fp_filtered):,}")
print(f"Отсечено (после дефолта):    {before_filter - len(fp_filtered):,}")

# Исключаем определяющие факторы из предикторов.
# Для каждой записи проверяем: является ли (fp_num, TYPE_FP) определяющим
# для данного сегмента.
def is_defining_factor(row):
    """Проверяет, входит ли фактор в DEFAULT_RULES для сегмента строки."""
    seg = row["segment"]
    rules = DEFAULT_RULES.get(seg, {})
    if row["TYPE_FP"] == "ФП" and row["fp_num"] in rules.get("FP", set()):
        return True
    if row["TYPE_FP"] == "СФП" and row["fp_num"] in rules.get("SFP", set()):
        return True
    return False

fp_filtered["_is_def"] = fp_filtered.apply(is_defining_factor, axis=1)
n_excluded = fp_filtered["_is_def"].sum()
fp_predictors = fp_filtered[~fp_filtered["_is_def"]].copy()

print(f"\nОпределяющих факторов исключено из предикторов: {n_excluded:,}")
print(f"ФП-предикторов для анализа:                     {len(fp_predictors):,}")
print(f"Уникальных типов ФП-предикторов:                {fp_predictors['fp_num'].nunique()}")

### 4.1 Потеря записей при временном фильтре

Оцениваем масштаб отсечения: **сколько записей ФП** мы теряем при фильтрации
«только до дефолта», в разрезе сегментов.

Это важно: если у большинства дефолтных компаний бóльшая часть ФП возникает
**после** дефолта, значит наша выборка предикторов будет бедной.

In [ ]:
# Только записи дефолтных компаний
fp_def_all = fp_all[fp_all["default_flag"] == 1].copy()
fp_def_before = fp_def_all[fp_def_all["dt"] < fp_def_all["ref_date"]]
fp_def_after  = fp_def_all[fp_def_all["dt"] >= fp_def_all["ref_date"]]

print("Записи ФП у дефолтных компаний:\n")
print(f"{'Сегмент':<8} {'Всего':<12} {'До дефолта':<14} {'После дефолта':<16} {'% отсечено'}")
print("=" * 62)

for seg in SEG_ORDER:
    n_all = len(fp_def_all[fp_def_all["segment"] == seg])
    n_before = len(fp_def_before[fp_def_before["segment"] == seg])
    n_after = len(fp_def_after[fp_def_after["segment"] == seg])
    pct = n_after / n_all * 100 if n_all > 0 else 0
    print(f"{seg:<8} {n_all:<12,} {n_before:<14,} {n_after:<16,} {pct:.1f}%")

n_all_t = len(fp_def_all)
n_before_t = len(fp_def_before)
n_after_t = len(fp_def_after)
pct_t = n_after_t / n_all_t * 100 if n_all_t > 0 else 0
print(f"{'ИТОГО':<8} {n_all_t:<12,} {n_before_t:<14,} {n_after_t:<16,} {pct_t:.1f}%")

# Среднее количество ФП до/после дефолта на одну компанию
print("\nСреднее кол-во записей ФП на одну дефолтную компанию:\n")
print(f"{'Сегмент':<8} {'До дефолта':<14} {'После дефолта'}")
print("=" * 38)
for seg in SEG_ORDER:
    n_def = companies[(companies["segment"] == seg) & (companies["default_flag"] == 1)].shape[0]
    if n_def == 0:
        continue
    avg_before = len(fp_def_before[fp_def_before["segment"] == seg]) / n_def
    avg_after  = len(fp_def_after[fp_def_after["segment"] == seg]) / n_def
    print(f"{seg:<8} {avg_before:<14.1f} {avg_after:.1f}")

## 5. Pivot (wide-формат)

Преобразуем «длинную» таблицу (1 строка = 1 событие ФП) в **широкую**
(1 строка = 1 компания, колонки = типы ФП).

- `aggfunc="sum"` — значения = **количество вхождений** каждого ФП
  (сколько раз данный ФП возникал у компании до дефолта)
- Дополнительно создаём **бинарную** версию (`0/1`) для корреляций и Lift

In [ ]:
# Количественный pivot
fp_predictors["value"] = 1
fp_wide = (
    fp_predictors
    .pivot_table(index="inn", columns="fp_num", values="value",
                 aggfunc="sum", fill_value=0)
    .reset_index()
)
fp_wide.columns.name = None

# Присоединяем мета-данные из реестра компаний
meta_cols = ["inn", "segment", "default_flag", "default_date"]
dataset = fp_wide.merge(companies[meta_cols], on="inn", how="left")
dataset["default_flag"] = dataset["default_flag"].fillna(0).astype("int8")

# Определяем колонки ФП (все кроме мета-данных)
fp_cols = [c for c in dataset.columns if c not in meta_cols]

# Бинарная версия: clip(upper=1) превращает количество в 0/1
dataset_bin = dataset.copy()
dataset_bin[fp_cols] = dataset_bin[fp_cols].clip(upper=1).astype("int8")

# Метрики по количеству ФП
dataset["fp_count"] = dataset[fp_cols].sum(axis=1)        # общее кол-во записей ФП
dataset["fp_unique"] = (dataset[fp_cols] > 0).sum(axis=1)  # уникальных типов ФП
dataset_bin["fp_count"] = dataset["fp_count"]
dataset_bin["fp_unique"] = dataset["fp_unique"]

# Компании, которые есть в реестре, но не попали в датасет
# (нет ФП-предикторов до дефолта)
companies_no_fp = companies[~companies["inn"].isin(dataset["inn"])].copy()

n_def_companies = int(companies["default_flag"].sum())
n_def_dataset = int(dataset["default_flag"].sum())
n_lost = n_def_companies - n_def_dataset

print(f"Итоговый датасет: {dataset.shape[0]:,} компаний × {len(fp_cols)} ФП-колонок")
print(f"  Дефолтных в датасете:    {n_def_dataset:,}")
print(f"  Вер. дефолта в датасете: {dataset['default_flag'].mean():.2%}")

print(f"\nПо сегментам:")
for seg in SEG_ORDER:
    s = dataset[dataset["segment"] == seg]
    n = len(s); nd = s["default_flag"].sum()
    print(f"  {seg}: {n:,} компаний, {nd:,} дефолтных ({nd/n*100:.2f}%)" if n > 0 else f"  {seg}: 0")

# ── Явный вывод: «внезапные» дефолты (нет ФП-предикторов до дефолта) ──
# Эти компании дефолтнули, но до даты дефолта у них нет ни одного ФП
# (кроме самого определяющего). Они не попадают в pivot и не участвуют
# в анализе предикторов.
def_no_fp = companies_no_fp[companies_no_fp["default_flag"] == 1]
nondef_no_fp = companies_no_fp[companies_no_fp["default_flag"] == 0]

print(f"\n{'='*70}")
print(f"КОМПАНИИ, НЕ ПОПАВШИЕ В ДАТАСЕТ (нет ФП-предикторов):")
print(f"  Дефолтных:    {len(def_no_fp):,} из {n_def_companies:,} "
      f"({len(def_no_fp)/n_def_companies*100:.1f}%) — 'внезапные' дефолты")
print(f"  Недефолтных:  {len(nondef_no_fp):,}")
if len(def_no_fp) > 0:
    print(f"\n  'Внезапные' дефолты по сегментам:")
    for seg in SEG_ORDER:
        n_seg_lost = len(def_no_fp[def_no_fp['segment'] == seg])
        n_seg_all = int(companies[(companies['segment']==seg) & (companies['default_flag']==1)].shape[0])
        if n_seg_lost > 0:
            print(f"    {seg}: {n_seg_lost:,} из {n_seg_all:,} "
                  f"({n_seg_lost/n_seg_all*100:.1f}%)")
    print(f"\n  У этих компаний определяющий фактор стал первым событием —")
    print(f"  предсказать их дефолт по предшествующим ФП невозможно.")

## 6. Гистограммы: количество ФП (дефолтные vs недефолтные)

Сравниваем распределение **числа уникальных ФП** у дефолтных и недефолтных компаний.

Если дефолтные в среднем имеют **больше** ФП — это подтверждает гипотезу,
что накопление факторов проблемности предшествует дефолту.

In [ ]:
# Статистика: среднее и медиана уникальных ФП
print("Число уникальных ФП-предикторов на компанию:\n")
print(f"  {'Сегмент':<20s} {'Дефолтные':>16s} {'Недефолтные':>16s}")
print("  " + "-" * 55)
for seg in SEG_ORDER + ["ВСЕ"]:
    s = dataset if seg == "ВСЕ" else dataset[dataset["segment"] == seg]
    d = s[s["default_flag"] == 1]["fp_unique"]
    nd = s[s["default_flag"] == 0]["fp_unique"]
    if len(d) > 0 and len(nd) > 0:
        print(f"  {seg:<20s} {d.mean():>6.1f} (med {d.median():>3.0f})  "
              f"{nd.mean():>6.1f} (med {nd.median():>3.0f})")

# Графики: нормализованные гистограммы (плотность) по сегментам
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plot_items = list(zip(SEG_ORDER + ["ВСЕ"], axes.flat))

for seg, ax in plot_items:
    s = dataset if seg == "ВСЕ" else dataset[dataset["segment"] == seg]
    d = s[s["default_flag"] == 1]["fp_unique"]
    nd = s[s["default_flag"] == 0]["fp_unique"]
    max_fp = int(s["fp_unique"].quantile(0.95)) + 1
    bins = range(0, max_fp + 2)
    ax.hist(nd.clip(upper=max_fp), bins=bins, alpha=0.7, density=True,
            label=f"Недеф. ({len(nd):,})", color="steelblue", edgecolor="white")
    ax.hist(d.clip(upper=max_fp), bins=bins, alpha=0.7, density=True,
            label=f"Деф. ({len(d):,})", color="tomato", edgecolor="white")
    ax.set_xlabel("Уникальных ФП")
    ax.set_ylabel("Плотность")
    ax.set_title(seg, fontweight="bold")
    ax.legend(fontsize=8)

plt.suptitle("Распределение числа ФП-предикторов: дефолтные vs недефолтные",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Динамика дефолтов по месяцам / кварталам

Визуализируем, **когда** (по месяцам и кварталам) компании входили в дефолт.
Это позволяет выявить сезонность и тренды.

In [ ]:
def_companies = companies[companies["default_flag"] == 1].copy()
def_companies["def_month"] = def_companies["default_date"].dt.to_period("M")
def_companies["def_quarter"] = def_companies["default_date"].dt.to_period("Q")

# ── По месяцам: дефолты ──
monthly = def_companies.groupby(["def_month", "segment"]).size().unstack(fill_value=0)
monthly = monthly[[s for s in SEG_ORDER if s in monthly.columns]]
monthly["Итого"] = monthly.sum(axis=1)
monthly.index = monthly.index.astype(str)

# ── По месяцам: общее количество ФП/СФП ──
df_with_month = df.copy()
df_with_month["month"] = df_with_month["dt"].dt.to_period("M")
fp_monthly = df_with_month.groupby(["month", "segment"]).size().unstack(fill_value=0)
fp_monthly = fp_monthly[[s for s in SEG_ORDER if s in fp_monthly.columns]]
fp_monthly["Итого"] = fp_monthly.sum(axis=1)
fp_monthly.index = fp_monthly.index.astype(str)

# Объединяем: дефолты + ФП в одну таблицу
combined_monthly = monthly.copy()
combined_monthly.columns = [f"Деф. {c}" for c in combined_monthly.columns]
for seg in SEG_ORDER:
    if seg in fp_monthly.columns:
        combined_monthly[f"ФП {seg}"] = fp_monthly[seg].reindex(monthly.index, fill_value=0)
combined_monthly["ФП Итого"] = fp_monthly["Итого"].reindex(monthly.index, fill_value=0)

print("Помесячная динамика: дефолты и общее количество ФП/СФП:")
display(combined_monthly)

# ── Графики ──
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
colors = {"МкБ": "#4472C4", "МСБ": "#ED7D31", "ККБ": "#70AD47"}

for seg in SEG_ORDER:
    if seg in monthly.columns:
        axes[0].plot(monthly.index, monthly[seg], marker="o", markersize=3,
                     label=seg, color=colors[seg], linewidth=1.5)
axes[0].set_title("Новые дефолты по месяцам", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Количество ИНН")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45, labelsize=7)

# По кварталам — stacked bar
quarterly = def_companies.groupby(["def_quarter", "segment"]).size().unstack(fill_value=0)
quarterly = quarterly[[s for s in SEG_ORDER if s in quarterly.columns]]
quarterly["Итого"] = quarterly.sum(axis=1)
quarterly.index = quarterly.index.astype(str)

seg_cols = [s for s in SEG_ORDER if s in quarterly.columns]
quarterly[seg_cols].plot(kind="bar", stacked=False, ax=axes[1],
                         color=[colors[s] for s in seg_cols], edgecolor="white", width=0.8)
axes[1].set_title("Новые дефолты по кварталам", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Количество ИНН")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Тепловая карта корреляций ФП (по сегментам)

Phi-корреляция (эквивалент Пирсона для бинарных признаков) между наиболее
частыми ФП. Показывает, **какие ФП часто встречаются вместе** у одних и тех же
компаний. Высокая корреляция может означать:
- ФП вызваны одной причиной (мультиколлинеарность)
- ФП являются частью одной «цепочки» событий

Строим **отдельно по сегментам**, т.к. наборы ФП различаются.

In [ ]:
TOP_N_HEATMAP = 30
corr_by_seg = {}

for seg in SEG_ORDER:
    subset = dataset_bin[dataset_bin["segment"] == seg]
    n_comp = len(subset)
    n_def = int(subset["default_flag"].sum())

    if n_comp < 10:
        print(f"{seg}: слишком мало компаний ({n_comp}) — пропуск")
        continue

    # Берём только ФП, встречающиеся хотя бы у 2 компаний
    fp_freq = subset[fp_cols].sum().sort_values(ascending=False)
    active_fp = fp_freq[fp_freq >= 2].head(TOP_N_HEATMAP).index.tolist()

    if len(active_fp) < 2:
        print(f"{seg}: менее 2 активных ФП — пропуск")
        continue

    corr = subset[active_fp].corr()
    mask_tri = np.triu(np.ones(corr.shape, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr, mask=mask_tri, annot=False, cmap="RdBu_r", center=0,
                vmin=-0.5, vmax=0.5, linewidths=0.3,
                xticklabels=True, yticklabels=True, ax=ax)
    ax.set_title(f"Корреляции ФП — {seg} ({n_comp:,} компаний, {n_def:,} дефолтных)",
                 fontsize=13, fontweight="bold")
    plt.xticks(fontsize=7, rotation=90)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.show()

    # Пары с сильной корреляцией
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack().reset_index()
    )
    pairs.columns = ["ФП_1", "ФП_2", "Корреляция"]
    strong = pairs[pairs["Корреляция"].abs() > 0.3].sort_values(
        "Корреляция", key=abs, ascending=False)
    corr_by_seg[seg] = strong
    if len(strong) > 0:
        print(f"  {seg}: пары с |r| > 0.3: {len(strong)}")
        display(strong.head(10).style.hide(axis="index").format({"Корреляция": "{:.3f}"}))
    else:
        print(f"  {seg}: пар с |r| > 0.3 нет")
    print()

## 9. Кросс-таблицы «ФП × Дефолт» и Lift

Для каждого ФП-предиктора рассчитываем:
- **Частота** — у скольких компаний есть этот ФП
- **Вер. дефолта** — какой % носителей ФП дефолтнул
- **Lift** = `(Вер. дефолта с ФП) / (базовая Вер. дефолта)` — во сколько раз дефолтность
  с данным ФП выше, чем в среднем

Lift > 1 → ФП «опасный», Lift < 1 → ФП «нейтральный» или «безопасный».

**Определяющие факторы исключены** — они не фигурируют в этих таблицах.

In [ ]:
def compute_cross_table(ds, fp_columns, label=""):
    """Считает частоту, вероятность дефолта и Lift для каждого ФП."""
    base_rate = ds["default_flag"].mean()
    total = len(ds)
    total_def = ds["default_flag"].sum()
    rows = []
    for col in fp_columns:
        n_with = (ds[col] == 1).sum()
        if n_with == 0:
            continue
        n_def_with = ds.loc[ds[col] == 1, "default_flag"].sum()
        rate = n_def_with / n_with if n_with > 0 else 0
        lift = rate / base_rate if base_rate > 0 else 0
        rows.append({
            "Тип ФП": col,
            "Компаний с ФП": n_with,
            "% от всех": round(n_with / total * 100, 2),
            "Дефолтов": n_def_with,
            "Вер. дефолта %": round(rate * 100, 2),
            "Lift": round(lift, 2),
        })
    ct = pd.DataFrame(rows).sort_values("Lift", ascending=False)
    return ct, base_rate, total, total_def


# ── Общая таблица ──
ct_all, br_all, n_all, nd_all = compute_cross_table(dataset_bin, fp_cols)
print(f"Общая: базовая вер. дефолта = {br_all:.2%}, компаний = {n_all:,}, дефолтов = {nd_all:,}")
print(f"\nТОП-30 ФП по Lift (≥3 дефолтов):")
display(
    ct_all[ct_all["Дефолтов"] >= 3].head(30)
    .style.hide(axis="index")
    .format({"Вер. дефолта %": "{:.2f}%", "% от всех": "{:.2f}%"})
    .bar(subset=["Lift"], color="salmon")
)

# ── По сегментам ──
cross_tables = {}
for seg in SEG_ORDER:
    seg_ds = dataset_bin[dataset_bin["segment"] == seg]
    seg_fp_cols = [c for c in fp_cols if (seg_ds[c] > 0).any()]
    ct, br, n, nd = compute_cross_table(seg_ds, seg_fp_cols, seg)
    cross_tables[seg] = ct
    print(f"\n{'='*70}")
    print(f"{seg}: базовая вер. дефолта = {br:.2%}, компаний = {n:,}, дефолтов = {nd:,}")
    print(f"ТОП-20 ФП по Lift (≥3 дефолтов):")
    filtered = ct[ct["Дефолтов"] >= 3]
    if len(filtered) > 0:
        display(
            filtered.head(20)
            .style.hide(axis="index")
            .format({"Вер. дефолта %": "{:.2f}%", "% от всех": "{:.2f}%"})
            .bar(subset=["Lift"], color="salmon")
        )
    else:
        print("  Нет ФП с ≥3 дефолтами")

## 10. Топ комбинаций (пары и тройки ФП) по Lift

Ищем **комбинации** ФП, которые совместно дают более высокий Lift,
чем каждый ФП по отдельности. Это может указывать на **синергию** факторов —
по отдельности они не опасны, но вместе резко повышают вероятность дефолта.

In [ ]:
MIN_COMPANIES_COMBO = 10  # минимум компаний с комбинацией
MIN_DEFAULTS_COMBO = 3    # минимум дефолтов (иначе lift случаен)
pairs_by_seg = {}
triples_by_seg = {}

for seg in SEG_ORDER:
    seg_ds = dataset_bin[dataset_bin["segment"] == seg]
    base_rate = seg_ds["default_flag"].mean()
    if base_rate == 0:
        print(f"{seg}: нет дефолтов — пропуск")
        continue

    y = seg_ds["default_flag"].values
    # Берём только достаточно частые ФП (≥20 компаний)
    seg_fp_cols = [c for c in fp_cols if (seg_ds[c] == 1).sum() >= 20]
    X = seg_ds[seg_fp_cols].values
    names = np.array(seg_fp_cols)

    # ── Пары ──
    pair_res = []
    for i, j in combinations(range(len(seg_fp_cols)), 2):
        mask = (X[:, i] == 1) & (X[:, j] == 1)
        n = mask.sum()
        if n < MIN_COMPANIES_COMBO:
            continue
        n_def = y[mask].sum()
        if n_def < MIN_DEFAULTS_COMBO:
            continue
        rate = n_def / n
        pair_res.append({
            "Комбинация": f"{names[i]}  +  {names[j]}",
            "Компаний": n, "Дефолтов": n_def,
            "Вер. деф. %": round(rate * 100, 2),
            "Lift": round(rate / base_rate, 2),
        })
    df_pairs = pd.DataFrame(pair_res).sort_values("Lift", ascending=False) if pair_res else pd.DataFrame()
    pairs_by_seg[seg] = df_pairs

    # ── Тройки (из топ-30 по индивидуальному Lift) ──
    seg_ct = cross_tables.get(seg, pd.DataFrame())
    top_individual = seg_ct[seg_ct["Компаний с ФП"] >= 20].head(30)["Тип ФП"].tolist()
    top_idx = [i for i, name in enumerate(seg_fp_cols) if name in top_individual]

    triple_res = []
    for i, j, k in combinations(top_idx, 3):
        mask = (X[:, i] == 1) & (X[:, j] == 1) & (X[:, k] == 1)
        n = mask.sum()
        if n < MIN_COMPANIES_COMBO:
            continue
        n_def = y[mask].sum()
        if n_def < MIN_DEFAULTS_COMBO:
            continue
        rate = n_def / n
        triple_res.append({
            "Комбинация": f"{names[i]}  +  {names[j]}  +  {names[k]}",
            "Компаний": n, "Дефолтов": n_def,
            "Вер. деф. %": round(rate * 100, 2),
            "Lift": round(rate / base_rate, 2),
        })
    df_triples = pd.DataFrame(triple_res).sort_values("Lift", ascending=False) if triple_res else pd.DataFrame()
    triples_by_seg[seg] = df_triples

    print(f"\n{'='*70}")
    print(f"{seg}: базовая вер. дефолта = {base_rate:.2%}, пар: {len(df_pairs)}, троек: {len(df_triples)}")

    if len(df_pairs) > 0:
        print(f"\n  ТОП-5 пар по Lift:")
        display(df_pairs.head(5).style.hide(axis="index")
                .format({"Вер. деф. %": "{:.2f}%"}).bar(subset=["Lift"], color="salmon"))
    if len(df_triples) > 0:
        print(f"\n  ТОП-5 троек по Lift:")
        display(df_triples.head(5).style.hide(axis="index")
                .format({"Вер. деф. %": "{:.2f}%"}).bar(subset=["Lift"], color="salmon"))

## 11. Среднее время между каждым ФП и дефолтом

Для каждого уникального ФП, зафиксированного у дефолтных компаний **до** даты дефолта,
рассчитываем временной промежуток (в днях) между датой появления ФП и датой дефолта.

Агрегируем по номеру ФП: **среднее**, **медиана**, **минимум**, **максимум** и **количество наблюдений**.

> **Важно**: ФП, встречающиеся менее чем у 5 дефолтных компаний, имеют статистически
> ненадёжные показатели — выводы по ним делать не следует. Такие факторы отмечены
> в таблице, но при интерпретации приоритет отдаётся ФП с достаточным количеством
> наблюдений (≥ 5).

In [ ]:
# Берём только ФП дефолтных компаний, возникшие ДО дефолта (fp_filtered уже
# содержит записи с dt < ref_date). Фильтруем по default_flag == 1.
fp_def_before = fp_filtered[fp_filtered["default_flag"] == 1].copy()

# Время (дни) от появления ФП до дефолта
fp_def_before["days_to_default"] = (
    fp_def_before["ref_date"] - fp_def_before["dt"]
).dt.days

# Агрегация по (сегмент, номер ФП, тип)
fp_time_stats = (
    fp_def_before
    .groupby(["segment", "fp_num", "TYPE_FP"])["days_to_default"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)
fp_time_stats.columns = [
    "Сегмент", "ФП", "Тип", "N записей", "Среднее (дни)",
    "Медиана (дни)", "Мин (дни)", "Макс (дни)"
]
fp_time_stats["Среднее (дни)"] = fp_time_stats["Среднее (дни)"].round(0).astype(int)
fp_time_stats["Медиана (дни)"] = fp_time_stats["Медиана (дни)"].round(0).astype(int)

# Подсчитываем уникальные ИНН (компании) для каждого ФП
fp_inn_count = (
    fp_def_before
    .groupby(["segment", "fp_num", "TYPE_FP"])["inn"]
    .nunique()
    .reset_index()
    .rename(columns={"inn": "Уник. комп."})
)
fp_time_stats = fp_time_stats.merge(
    fp_inn_count, left_on=["Сегмент", "ФП", "Тип"],
    right_on=["segment", "fp_num", "TYPE_FP"], how="left"
).drop(columns=["segment", "fp_num", "TYPE_FP"])

# Перемещаем "Уник. комп." ближе к началу
cols = ["Сегмент", "ФП", "Тип", "N записей", "Уник. комп.",
        "Среднее (дни)", "Медиана (дни)", "Мин (дни)", "Макс (дни)"]
fp_time_stats = fp_time_stats[cols]

MIN_COMPANIES_RELIABLE = 5

for seg in SEG_ORDER:
    seg_data = fp_time_stats[fp_time_stats["Сегмент"] == seg].copy()
    reliable = seg_data[seg_data["Уник. комп."] >= MIN_COMPANIES_RELIABLE]
    unreliable = seg_data[seg_data["Уник. комп."] < MIN_COMPANIES_RELIABLE]

    print(f"\n{'='*80}")
    print(f"{seg}: ФП дефолтных компаний — время до дефолта")
    print(f"{'='*80}")
    print(f"Всего уникальных ФП: {len(seg_data)}")
    print(f"  Надёжных (≥{MIN_COMPANIES_RELIABLE} комп.): {len(reliable)}")
    print(f"  Ненадёжных (<{MIN_COMPANIES_RELIABLE} комп.): {len(unreliable)}")

    if len(reliable) > 0:
        print(f"\nТоп-20 ФП с наименьшим временем до дефолта (≥{MIN_COMPANIES_RELIABLE} комп.):")
        bottom = reliable.sort_values("Медиана (дни)", ascending=True).head(20)
        display(
            bottom.drop(columns=["Сегмент"])
            .style.hide(axis="index")
            .background_gradient(subset=["Медиана (дни)"], cmap="YlOrRd_r")
        )

print(f"\n{'='*80}")
print("ПРИМЕЧАНИЕ: ФП, встречающиеся менее чем у 5 дефолтных компаний,")
print("не включены в рейтинги — их статистика ненадёжна из-за малого")
print(f"размера выборки. Всего таких ФП: "
      f"{len(fp_time_stats[fp_time_stats['Уник. комп.'] < MIN_COMPANIES_RELIABLE])}")
print(f"{'='*80}")

## 12. Аномалии и паттерны

Выявляем нетипичные случаи:
1. **ФП со 100% дефолтностью** — все компании с этим ФП дефолтнули
2. **ФП с 0% дефолтностью** — ни один носитель не дефолтнул (при ≥20 компаниях)
3. **Редкие ФП** — по ним нельзя делать выводы (< 5 компаний)
4. **Компании-выбросы** — аномально много ФП
5. **Дефолтные без предикторов** — ни одного ФП до дефолта

In [ ]:
print("="*70)
print("АНОМАЛИИ И ПАТТЕРНЫ")
print("="*70)

fp_100 = ct_all[(ct_all["Вер. дефолта %"] == 100) & (ct_all["Компаний с ФП"] >= 2)]
print(f"\n1. ФП с 100% дефолтностью (≥2 компании): {len(fp_100)}")
if len(fp_100) > 0:
    display(fp_100.style.hide(axis="index"))

fp_0 = ct_all[(ct_all["Вер. дефолта %"] == 0) & (ct_all["Компаний с ФП"] >= 20)]
print(f"\n2. ФП с 0% дефолтностью (≥20 компаний): {len(fp_0)}")
if len(fp_0) > 0:
    display(fp_0.style.hide(axis="index"))

rare = ct_all[ct_all["Компаний с ФП"] < 5]
print(f"\n3. Редкие ФП (< 5 компаний): {len(rare)} из {len(ct_all)}")

q99 = dataset["fp_unique"].quantile(0.99)
outliers = dataset[dataset["fp_unique"] > q99]
all_outliers = outliers.copy()
print(f"\n4. Компании-выбросы (> {q99:.0f} ФП, 99-й перцентиль): {len(outliers)}")
if len(outliers) > 0:
    display(
        outliers[["inn", "segment", "default_flag", "fp_unique", "fp_count"]]
        .sort_values("fp_unique", ascending=False)
        .head(10).style.hide(axis="index")
    )

def_no_fp = companies_no_fp[companies_no_fp["default_flag"] == 1]
print(f"\n5. Дефолтные без ФП-предикторов до дефолта: {len(def_no_fp)}")
if len(def_no_fp) > 0:
    print(f"   ({len(def_no_fp)/n_def_companies:.1%} от всех дефолтных)")
    for seg in SEG_ORDER:
        n = len(def_no_fp[def_no_fp["segment"] == seg])
        if n > 0:
            print(f"     {seg}: {n}")

In [ ]:
# ── Аномалии по сегментам ──
high_lift_by_seg = {}
for seg in SEG_ORDER:
    ct = cross_tables.get(seg)
    if ct is None or len(ct) == 0:
        continue
    seg_100 = ct[(ct["Вер. дефолта %"] == 100) & (ct["Компаний с ФП"] >= 2)]
    high_lift = ct[(ct["Lift"] > 2) & (ct["Дефолтов"] >= 3)]
    high_lift_by_seg[seg] = high_lift
    print(f"\n{seg}:")
    print(f"  ФП с 100% вер. дефолта (≥2 комп.): {len(seg_100)}")
    print(f"  ФП с Lift > 2 (≥3 деф.): {len(high_lift)}")
    if len(high_lift) > 0:
        display(high_lift.head(10).style.hide(axis="index")
                .format({"Вер. дефолта %": "{:.2f}%", "% от всех": "{:.2f}%"}))

## 13. Сводка, гипотезы и сохранение

In [ ]:
print("\u2554" + "\u2550"*78 + "\u2557")
print("\u2551" + " СВОДКА ПО РЕЗУЛЬТАТАМ АНАЛИЗА".center(78) + "\u2551")
print("\u2560" + "\u2550"*78 + "\u2563")

for seg in SEG_ORDER:
    seg_c = companies[companies["segment"] == seg]
    n_total = len(seg_c)
    n_def = seg_c["default_flag"].sum()
    pct = n_def / n_total * 100 if n_total > 0 else 0
    rules = DEFAULT_RULES[seg]
    ct = cross_tables.get(seg, pd.DataFrame())
    n_high_lift = len(ct[(ct["Lift"] > 2) & (ct["Дефолтов"] >= 3)]) if len(ct) > 0 else 0
    seg_ds = dataset[dataset["segment"] == seg]
    avg_def = seg_ds[seg_ds["default_flag"]==1]["fp_unique"].mean() if n_def > 0 else 0
    avg_nodef = seg_ds[seg_ds["default_flag"]==0]["fp_unique"].mean()
    print(f"\u2551 {seg}:".ljust(79) + "\u2551")
    print(f"\u2551   Правило: FP={sorted(rules['FP'])} | SFP={sorted(rules['SFP'])}".ljust(79) + "\u2551")
    print(f"\u2551   ИНН: {n_total:,}  |  Дефолтных: {n_def:,} ({pct:.1f}%)".ljust(79) + "\u2551")
    print(f"\u2551   Ср. ФП: деф={avg_def:.1f}, недеф={avg_nodef:.1f}  |  ФП с Lift>2: {n_high_lift}".ljust(79) + "\u2551")
    print("\u2551" + " "*78 + "\u2551")
print("\u255a" + "\u2550"*78 + "\u255d")


In [ ]:
out_dir = DATA_DIR
dataset.to_pickle(f"{out_dir}/dataset_default_logic.pkl")

excel_path = f'{out_dir}/Приложение к отчету "Анализ дефолтов (логика ДМУКП)".xlsx'
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    ct_all.to_excel(writer, sheet_name="Кросс-таблица (общая)", index=False)
    for seg in SEG_ORDER:
        ct = cross_tables.get(seg)
        if ct is not None and len(ct) > 0:
            ct.to_excel(writer, sheet_name=f"Кросс-таблица ({seg})", index=False)
    if len(fp_100) > 0:
        fp_100.to_excel(writer, sheet_name="100% дефолтность", index=False)
    if len(fp_0) > 0:
        fp_0.to_excel(writer, sheet_name="0% дефолтность", index=False)

    # Корреляции ФП
    for seg, df_c in corr_by_seg.items():
        if len(df_c) > 0:
            df_c.to_excel(writer, sheet_name=f"Корреляции ({seg})"[:31], index=False)

    # Комбинации пар и троек
    for seg, df_p in pairs_by_seg.items():
        if len(df_p) > 0:
            df_p.to_excel(writer, sheet_name=f"Пары ФП ({seg})"[:31], index=False)
    for seg, df_t in triples_by_seg.items():
        if len(df_t) > 0:
            df_t.to_excel(writer, sheet_name=f"Тройки ФП ({seg})"[:31], index=False)

    # Аномалии
    if len(all_outliers) > 0:
        all_outliers.to_excel(writer, sheet_name="Аномалии-выбросы", index=False)
    for seg, df_hl in high_lift_by_seg.items():
        if len(df_hl) > 0:
            df_hl.to_excel(writer, sheet_name=f"Аномалии Lift ({seg})"[:31], index=False)

print(f"Сохранено:")
print(f"  {out_dir}/dataset_default_logic.pkl")
print(f"  {excel_path}")
